<a id="setup"></a>
# Diagnostic de la tokenisation actuelle sur données réelles

Objectif : comprendre précisément où et pourquoi SentencePiece v1 (entraîné uniquement sur Bayelemabaga, module 07) échoue sur le vrai bambara de CorBaMa — ce diagnostic a justifié la création de la v2 (déjà entraînée dans le module 07).

In [1]:
import sys, glob
from pathlib import Path

sys.path.append(str(Path.cwd().parent))
from bambara_pos_utils import *

import sentencepiece as spm
from collections import Counter

sp = spm.SentencePieceProcessor(
    model_file=str(Path.cwd().parent / "07_sentencepiece_tokenization" / "bambara_sp.model")
)

corbama_data = load_bambara_corpus(
    Path.cwd().parent / "08_real_data_evaluation" / "corbama_corpus.tsv"
)
print(f"Corpus CorBaMa : {len(corbama_data)} phrases")

Corpus chargé : 166361 phrases.
Corpus CorBaMa : 166361 phrases


<a id="fragmentation"></a>
# Mots les plus fragmentés

In [2]:
fragmentation = Counter()
for words, _ in corbama_data:
    for w in words:
        n_pieces = len(sp.encode(w, out_type=str))
        if n_pieces >= 3:
            fragmentation[w] = n_pieces

print("Mots découpés en 3 sous-tokens ou plus (les 20 plus fragmentés) :")
for word, n_pieces in sorted(fragmentation.items(), key=lambda x: -x[1])[:20]:
    pieces = sp.encode(word, out_type=str)
    print(f"  {word:<18} {n_pieces} morceaux -> {pieces}")

Mots découpés en 3 sous-tokens ou plus (les 20 plus fragmentés) :
  9.10.11.12.13.14.15.16.17.18.19.20.21.22.23.24.25.26.27.28.29.30.31.1 48 morceaux -> ['▁', '9', '.', '10', '.', '11', '.', '12', '.', '13', '.', '14', '.', '15', '.', '16', '.', '17', '.', '18', '.', '19', '.', '20', '.', '21', '.', '22', '.', '23', '.', '24', '.', '25', '.', '26', '.', '27', '.', '28', '.', '29', '.', '30', '.', '31', '.', '1']
  10.11.12.13.14.15.16.17.18.19.20.21.22.23.24.25.26.27.28.29.30.1 44 morceaux -> ['▁', '10', '.', '11', '.', '12', '.', '13', '.', '14', '.', '15', '.', '16', '.', '17', '.', '18', '.', '19', '.', '20', '.', '21', '.', '22', '.', '23', '.', '24', '.', '25', '.', '26', '.', '27', '.', '28', '.', '29', '.', '30', '.', '1']
  Comité-Militaire-de-Libération-Nationale 28 morceaux -> ['▁C', 'o', 'mi', 't', 'é-', 'M', 'i', 'li', 'ta', 'i', 're', '-', 'de', '-', 'L', 'i', 'b', 'é', 'ra', 'ti', 'on', '-', 'N', 'a', 'ti', 'o', 'na', 'le']
  nègè-tè-bin-n-fari-kan-janko-a-ka-n-tigè

<a id="morpheme-comparison"></a>
# Comparaison avec la vraie décomposition morphologique

CorBaMa fournit, pour les mots composés, leur vraie décomposition en morphèmes (validée par des linguistes). On compare cette vérité à ce que SentencePiece v1 produit.

In [3]:
html_files = glob.glob(
    str(Path.cwd().parent / "08_real_data_evaluation" / "corbama-disamb" / "**" / "*.dis.html"),
    recursive=True
)

all_word_morpheme_pairs = []
for fp in html_files:
    all_word_morpheme_pairs.extend(get_word_morpheme_pairs(fp))

print(f"Mots avec décomposition morphologique connue : {len(all_word_morpheme_pairs)}")

Mots avec décomposition morphologique connue : 398582


In [4]:
print(f"{'Mot':<18}{'Vrais morphèmes':<35}{'SentencePiece v1'}")
print("-" * 80)
for word, true_morphemes in all_word_morpheme_pairs[:20]:
    sp_pieces = sp.encode(word, out_type=str)
    print(f"{word:<18}{str(true_morphemes):<35}{sp_pieces}")

Mot               Vrais morphèmes                    SentencePiece v1
--------------------------------------------------------------------------------
minnu             ['mîn', 'lu']                     ['▁minnu']
duguw             ['dùgu', 'w']                     ['▁duguw']
kitabunin         ['kìtabu', 'nin']                 ['▁kitabu', 'nin']
kunnafoni         ['kùn', 'lá', 'fóni']           ['▁kunnafoni']
banaw             ['bàna', 'w']                     ['▁banaw']
minw              ['mîn', 'w']                      ['▁min', 'w']
caya              ['cá', 'ya']                      ['▁caya']
kònòboli        ['kɔ́nɔ', 'bòli']                 ['▁k', 'ò', 'n', 'ò', 'boli']
finyèbanaw       ['fíɲɛ', 'bàna']                 ['▁fin', 'y', 'è', 'bana', 'w']
nyèdimi          ['ɲɛ́', 'dími']                   ['▁n', 'y', 'è', 'dimi']
tuguci            ['tùgu', 'cì']                   ['▁tugu', 'ci']
jòyòròba       ['jɔ̀yɔrɔ', 'jɔ̀', 'yɔ́rɔ', 'ba']  ['▁', 'j', 'ò', 'y'

In [5]:
diffs = []
for word, true_morphemes in all_word_morpheme_pairs:
    sp_pieces = sp.encode(word, out_type=str)
    diffs.append(abs(len(sp_pieces) - len(true_morphemes)))

avg_diff = sum(diffs) / len(diffs)
exact_match = sum(1 for d in diffs if d == 0) / len(diffs)
print(f"Écart moyen (nombre de morceaux) SentencePiece v1 vs vraie morphologie : {avg_diff:.2f}")
print(f"Correspondance exacte du nombre de morceaux : {exact_match*100:.1f}%")

Écart moyen (nombre de morceaux) SentencePiece v1 vs vraie morphologie : 1.20
Correspondance exacte du nombre de morceaux : 21.8%


# Interprétation

Un écart moyen élevé confirme que SentencePiece v1, entraîné uniquement sur les statistiques de Bayelemabaga, ne retrouve pas toujours la vraie structure morphologique du bambara. C'est ce diagnostic qui a motivé la v2 (déjà entraînée dans le module 07, guidée par les morphèmes réels de CorBaMa). Le notebook suivant compare les deux versions.